Buat EDA

In [2]:
import pandas as pd
from pathlib import Path

output modelnya bilang teddy istri pak presiden jir, mau di cek apa di dataset ada yang ngomong gitu

In [2]:
import json

input_file = "datasetSFT.jsonl"
output_file = "hasil_teddy.jsonl"
search_keyword = "teddy"

count = 0

# print(9500; Memulai pencarian kata '{search_keyword}'...")

# Buka file input untuk dibaca dan file output untuk ditulis
with open(input_file, "r", encoding="utf-8") as infile, \
     open(output_file, "w", encoding="utf-8") as outfile:
    
    for line in infile:
        if not line.strip():
            continue  # Lewati baris kosong jika ada
            
        # Cari kata "teddy" secara langsung di teks baris mentah (case-insensitive)
        if search_keyword.lower() in line.lower():
            outfile.write(line)
            count += 1

print(f"---")
print(f"&#10004; Selesai! Berhasil menemukan {count} data.")
print(f"&#128190; Hasilnya sudah disimpan ke: {output_file}")

---
&#10004; Selesai! Berhasil menemukan 87 data.
&#128190; Hasilnya sudah disimpan ke: hasil_teddy.jsonl


In [22]:
data_gabungan = pd.read_csv("csv_gabungan.csv")

In [23]:
data_gabungan["label"].unique()

array(['Fitnah', 'Disinformasi', 'Fakta', 'Bukan DFK', 'Ujaran Kebencian',
       'Netral', 'netral', 'ujaran_kebencian'], dtype=object)

secara label udah bener

coba kita cek pas udah dibentuk jadi format alpaca

In [24]:
data_SFT = pd.read_json("datasetSFT.jsonl", lines=True)

In [25]:
data_SFT.head()

,instruction,input,output
0,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'Guru Besar UGM tuduh kepala daerah ...,Penjelasan: Pernyataan Prof. Wahyudi dipelinti...
1,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'RT ini nih yg mau d panggil yg terh...,Penjelasan: Komentar merupakan ekspresi ketida...
2,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'Camat Kepanjenkidul Kota Blitar Ind...,Penjelasan: Ini adalah modus pencatutan nama y...
3,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'Gubernur Sulsel cuma mau cari popul...,Penjelasan: Claim tersebut menggunakan framing...
4,Anda adalah sistem ahli klasifikasi komentar m...,Komentar: 'Judul: .\n\nTeks: Warga Madura dike...,Penjelasan:\n* Claim menggeneralisir warga Mad...


In [26]:
data_SFT['label'] = data_SFT['output'].str.extract(r'Label:\s*(.*)', expand=False).str.strip()
data_SFT['label'] = data_SFT['label'].str.rstrip('.')


In [27]:
data_SFT['label'].unique()

array(['Fitnah', 'Netral', 'Disinformasi', 'Ujaran Kebencian'],
      dtype=object)

udah bener juga, sekalian cek dalam bentuk TRL

In [28]:
data_SFT_TRL = pd.read_json("datasetSFT_TRL.jsonl", lines=True)

In [29]:
data_SFT_TRL.head()

,messages
0,"[{'role': 'system', 'content': 'Anda adalah si..."
1,"[{'role': 'system', 'content': 'Anda adalah si..."
2,"[{'role': 'system', 'content': 'Anda adalah si..."
3,"[{'role': 'system', 'content': 'Anda adalah si..."
4,"[{'role': 'system', 'content': 'Anda adalah si..."


In [30]:
import pandas as pd
import json

data_list = []
with open("datasetSFT_TRL.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data_json = json.loads(line)
        
        assistant_content = ""
        for msg in data_json["messages"]:
            if msg["role"] == "assistant":
                assistant_content = msg["content"]
                break
        
        data_list.append({"assistant_response": assistant_content})

df_trl = pd.DataFrame(data_list)

# Ekstrak Label menggunakan Regex dari teks assistant
# Mencari kata 'Label:' lalu mengambil kata setelahnya, dan menghapus titik (.) di akhir
df_trl['label'] = df_trl['assistant_response'].str.extract(r'Label:\s*(.*)', expand=False).str.strip()
df_trl['label'] = df_trl['label'].str.rstrip('.')

# Cetak hasil untuk memastikan
print("Hasil Ekstraksi Label:")
print(df_trl['label'].value_counts()) # Melihat distribusi label yang berhasil diambil
print("\nContoh beberapa baris teratas:")
print(df_trl[['label']].head())

Hasil Ekstraksi Label:
label
Netral              7500
Fitnah              6341
Disinformasi        6341
Ujaran Kebencian    6341
Name: count, dtype: int64

Contoh beberapa baris teratas:
              label
0            Fitnah
1            Netral
2      Disinformasi
3            Fitnah
4  Ujaran Kebencian


In [31]:
df_trl

,assistant_response,label
0,Penjelasan: Pernyataan Prof. Wahyudi dipelinti...,Fitnah
1,Penjelasan: Komentar merupakan ekspresi ketida...,Netral
2,Penjelasan: Ini adalah modus pencatutan nama y...,Disinformasi
3,Penjelasan: Claim tersebut menggunakan framing...,Fitnah
4,Penjelasan:\n* Claim menggeneralisir warga Mad...,Ujaran Kebencian
...,...,...
26518,Penjelasan:\n* Claim menggeneralisir perempuan...,Ujaran Kebencian
26519,Penjelasan: Ini adalah modus penipuan yang ber...,Disinformasi
26520,Penjelasan: Ini adalah modus fabrikasi gambar ...,Disinformasi
26521,Penjelasan: Komentar ini bersifat opini dan ti...,Netral


In [33]:
df_trl["label"].value_counts()

label
Netral              7500
Fitnah              6341
Disinformasi        6341
Ujaran Kebencian    6341
Name: count, dtype: int64

# Cek label Unknown, hasil prediksi model

ini hasil prediksi sama ground truthnya, mau ku cek

In [36]:
pd.set_option('display.max_colwidth', None)

In [48]:
hasil_generated = pd.read_csv("Dataset_hasil/hasil_evaluasi_test_v1_full_320.csv")
len(hasil_generated)

2605

In [40]:
hasil_generated.head(1)

,input,true_output,true_label,true_reasoning,pred_output,pred_label,pred_reasoning,is_correct,bertscore_precision,bertscore_recall,bertscore_f1
0,Komentar: 'Jam lembur malah pada ngentot';,"Penjelasan: Komentar ini hanya berupa ekspresi frustrasi kasual dengan bahasa vulgar tanpa menargetkan kelompok mana pun\nBerikut adalah beberapa poin penting:\n* Komentar menggunakan kata vulgar untuk mengeluh tentang situasi kerja\n* Tidak menyerang identitas SARA atau individu tertentu\n* Sesuai aturan, umpatan seksual tanpa unsur kebencian dikategorikan netral\n* Tidak ada klaim informasi yang perlu diverifikasi\nLabel: Netral.",Netral,"Penjelasan: Komentar ini hanya berupa ekspresi frustrasi kasual dengan bahasa vulgar tanpa menargetkan kelompok mana pun\nBerikut adalah beberapa poin penting:\n* Komentar menggunakan kata vulgar untuk mengeluh tentang situasi kerja\n* Tidak menyerang identitas SARA atau individu tertentu\n* Sesuai aturan, umpatan seksual tanpa unsur kebencian dikategorikan netral\n* Tidak ada klaim informasi yang perlu diverifikasi",Penjelasan: Komentar merupakan guyonan kasual tanpa klaim fakta maupun unsur penghinaan kelompok\nBerikut adalah beberapa poin penting:\n* Mengamati isi kalimat sebagai ungkapan slang vulgar tanpa konteks informatif\n* Memeriksa tidak adanya target identitas SARA atau serangan terhadap kelompok tertentu\n* Menyimpulkan hanya ekspresi sarkasme ringan yang umum di internet\nLabel: Netral.,Netral,Penjelasan: Komentar merupakan guyonan kasual tanpa klaim fakta maupun unsur penghinaan kelompok\nBerikut adalah beberapa poin penting:\n* Mengamati isi kalimat sebagai ungkapan slang vulgar tanpa konteks informatif\n* Memeriksa tidak adanya target identitas SARA atau serangan terhadap kelompok tertentu\n* Menyimpulkan hanya ekspresi sarkasme ringan yang umum di internet,True,0.792192,0.788344,0.790263


In [41]:
hasil_generated["pred_label"].unique()

array(['Netral', 'Ujaran Kebencian', 'Fitnah', 'Disinformasi', 'Unknown'],
      dtype=object)

In [45]:
unknown = hasil_generated[hasil_generated["pred_label"] == "Unknown"] 
len(unknown)

10

In [47]:
for i, data in unknown.iterrows():
    print(len(data["input"]))

9251
54
74
6748
11462
11819
174
28
7796
5478


In [50]:
unknown.head(1)

input  \
82  Komentar: 'Arab Saudi sengaja mengirim Kiswah Ka'bah ke Epstein untuk mendanai kejahatan seksualnya!';\n\nArtikel Rujukan: Hal ini terungkap dalam berkas Epstein yang dirilis Departemen Hukum Amerika Serikat (DOJ) akhir Januari lalu.\nBerdasarkan pertukaran surat elektronik (email) dari 2017, menunjukkan pengiriman tiga potong kain yang diklaim sebagai Kiswah dari Arab Saudi ke rumah Epstein di Karibia\nBayangkan mengirimkan kain dari tempat paling suci di Bumi ke tempat paling kotor!" tulis seorang pengguna X.\nDalam foto bertarikh 2014, Epstein dan seorang pria tampak sedang memeriksa sepotong kain di tanah, yang menyerupai bagian paling berornamen dari Kiswah yang menutupi pintu masuk Ka'bah.\nPengguna X lainnya mengatakan foto itu "menghancurkan hati saya berkeping-keping" karena Kiswah itu "dibentangkan di lantai seperti karpet".\nNamun, foto ini tampaknya tidak terkait dengan dokumen yang menunjukkan potongan-potongan Kiswah dikirim ke Epstein pada 2017. Dan tidak jelas apakah kain dalam foto itu adalah potongan Kiswah asli.\nDibuat dari sutra hitam, Kiswah dihiasi dengan sulaman ayat-ayat Al-Quran menggunakan benang emas dan perak. Kiswah menyelimuti keempat dinding luar Ka'bah di Mekah.\nSetiap tahun, setelah disentuh oleh jutaan jamaah haji, Kiswah diganti dengan yang baru selama upacara perayaan Tahun Baru Islam.\nArsip DOJ berisi korespondensi antara staf Epstein dan akun email atas nama "Aziza al-Ahmadi", orang yang tampaknya mengatur pengiriman tiga bagian Kiswah ke Epstein pada 2017.\nSatu digambarkan berwarna hijau, berasal dari bagian dalam Ka'bah; kain hitam dari lapisan luar yang digunakan; dan ukiran bordir yang terbuat dari bahan yang sama, tetapi tidak digunakan.\nSebuah email tertanggal 1 Februari 2017, menunjukkan seorang asisten dari orang yang disebut Ahmadi memberitahu staf Epstein bahwa mereka akan "mengirimkan beberapa potongan kain Ka'bah untuk masjid". Tidak jelas apakah "masjid" yang dimaksud merujuk pada lokasi di properti Epstein.\nDari berkas-berkas Epstein yang telah ditinjau sejauh ini, tidak ada yang menyebutkan tentang masjid di pulau tersebut. Namun, menurut dokumen yang diunggah di situs web DOJ, ada referensi tentang bangunan kecil sebagai "kuil".\nTidak jelas apa yang dimaksud dalam korespondensi tersebut. Kuil di Little St James Island berupa bangunan kecil di bagian selatan pulau dengan kubah emas.\n"Masjid" yang disebutkan dalam dokumen tidak dapat disamakan dengan Masjidil Haram di Mekah.\nDokumen menunjukkan, barang kiriman tiba di rumah Epstein di Palm Beach pada 4 Maret 2017. Paket itu kemudian dikirim ke St Thomas di Kepulauan Virgin AS.\nLokasinya dekat dengan Little St James, pulau pribadi Epstein. Beberapa korban kekerasan seksual Epstein mengklaim bahwa mereka dibawa ke sana dan disiksa.\nPada formulir Bea Cukai AS tertanggal 14 Maret 2017, kiriman tersebut dinyatakan sebagai "lukisan, gambar, dan patung" dengan nilai US$10.980 (Rp183 juta).\nSebuah email tertanggal 21 Maret mengonfirmasi pengiriman potongan Kiswah ke "rumah Epstein".\nDari Trump, Elon Musk hingga Bill Gates Adakah orang Indonesia dalam dokumen Epstein terbaru?\nAndrew kehilangan gelar 'pangeran' UK akibat skandal seks\nTrump dalam pusaran berkas Jeffrey Epstein Apakah benar Presiden AS terlibat?\nSetelah pengiriman, email yang dikirim dari akun Ahmadi memberitahu Epstein bahwa kain hitam tersebut "telah disentuh oleh minimal 10 juta Muslim dari berbagai denominasi, Sunni, Syiah, dan lainnya".\n"Mereka berjalan mengelilingi Ka'bah tujuh kali, lalu semua orang berusaha sekuat tenaga untuk menyentuhnya dan mereka menaruh doa, harapan, air mata, dan impian mereka pada kain ini," menurut email tersebut.\nTidak jelas apakah Epstein menerima potongan Kiswah tersebut sebagai hadiah atau apakah potongan tersebut asli.\nNamun, ini bukanlah pengiriman pertama dari Arab Saudi ke Epstein, menurut berkas yang dirilis Departemen Kehakiman AS (DOJ).\nDalam rangkaian email pada 27 Januari 2017, orang yang didu